
# Transfer WV3 ARNet to the Local 6-Band Dataset

## الهدف

نقل أفضل ARNet مدرب على PanCollection WorldView-3 من:

```text
8 MS Bands + PAN
```

إلى بياناتك:

```text
6 MS Bands + PAN
```

ثم Fine-tuning على التقسيم المكاني المحلي:

```text
Train = 231
Validation = 22
Test = 22
```

## التحويل

```text
Head Conv:
9 input channels → 7 input channels

Tail Conv:
8 output bands → 6 output bands
```

## التدريب

```text
Stage A:
Head + Tail فقط

Stage B:
النموذج كاملًا
```

## إضافة Misalignment Augmentation

أثناء التدريب فقط، يتم تحريك PAN حتى `±2` بكسل عالي الدقة بدون Circular Wrap، بينما يظل Target ثابتًا.

## المقارنة النهائية

```text
Bicubic
Fusion CNN
WV3-Transfer HAT
WV3-Transfer ARNet
```


In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("فعّل T4 GPU.")

device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

from pathlib import Path
import sys
import shutil
import subprocess

%cd /content

REPO_DIR = Path("/content/ARConv")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/Xueyangwang-will/ARConv.git",
        str(REPO_DIR),
    ],
    check=True,
)

!pip install -q scikit-image pandas matplotlib tqdm

sys.path.insert(0, str(REPO_DIR / "models"))

from models import ARNet

print("Official ARNet imported.")


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

PATCHES_DIR = PROJECT_DIR / "Wald_Data_GSD" / "Patches"

TRAIN_DIR = PATCHES_DIR / "train"
VAL_DIR = PATCHES_DIR / "val"
TEST_DIR = PATCHES_DIR / "test"

STATS_PATH = (
    PROJECT_DIR
    / "Fusion_Baseline_Results"
    / "train_normalization_stats.json"
)

WV3_ARNET_PATH = (
    PROJECT_DIR
    / "WV3_ARNet_Official"
    / "best_wv3_arnet.pth"
)

EXPERT_TEST_CACHE = (
    PROJECT_DIR
    / "Spatial_OOF_HAT_CNN_Fusion"
    / "final_expert_predictions"
    / "test"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "WV3_Transfer_ARNet_6Band"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = OUTPUT_DIR / "best_wv3_transfer_arnet_6band.pth"
LAST_PATH = OUTPUT_DIR / "last_wv3_transfer_arnet_6band.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
TEST_JSON_PATH = OUTPUT_DIR / "test_metrics_comparison.json"
TEST_CSV_PATH = OUTPUT_DIR / "test_metrics_comparison.csv"

for path in [
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    STATS_PATH,
    WV3_ARNET_PATH,
    EXPERT_TEST_CACHE,
]:
    print(path, "→", path.exists())

    if not path.exists():
        raise FileNotFoundError(path)


In [ ]:

import os
import json
import math
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

SEED = 42

BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
NUM_WORKERS = 0

STAGE_A_EPOCHS = 5
STAGE_B_EPOCHS = 20

STAGE_A_LR = 1e-4
STAGE_B_LR = 2e-5

WEIGHT_DECAY = 1e-6
EARLY_STOPPING_PATIENCE = 6

HW_RANGE = [1, 9]

USE_MISALIGNMENT_AUGMENTATION = True
MISALIGNMENT_PROBABILITY = 0.5
MAX_SHIFT_HR = 2

# Default assumption: local six bands correspond to the first six WV3 bands.
# Change only when exact sensor-band correspondence is known.
WV3_TO_LOCAL_BAND_MAPPING = [0, 1, 2, 3, 4, 5]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Effective batch:", BATCH_SIZE * ACCUMULATION_STEPS)
print("Band mapping:", WV3_TO_LOCAL_BAND_MAPPING)
print("Misalignment:", USE_MISALIGNMENT_AUGMENTATION)


In [ ]:

with open(STATS_PATH, "r", encoding="utf-8") as file:
    stats = json.load(file)

MS_LOW = np.array(
    [item["p01"] for item in stats["ms"]],
    dtype=np.float32,
)[:, None, None]

MS_HIGH = np.array(
    [item["p99"] for item in stats["ms"]],
    dtype=np.float32,
)[:, None, None]

PAN_LOW = np.float32(stats["pan"]["p01"])
PAN_HIGH = np.float32(stats["pan"]["p99"])


def normalize_ms(array):
    return np.clip(
        (array.astype(np.float32) - MS_LOW)
        / np.maximum(MS_HIGH - MS_LOW, 1e-6),
        0,
        1,
    )


def normalize_pan(array):
    return np.clip(
        (array.astype(np.float32) - PAN_LOW)
        / max(float(PAN_HIGH - PAN_LOW), 1e-6),
        0,
        1,
    )


def tensor_to_dn(tensor):
    low = torch.from_numpy(MS_LOW).to(
        tensor.device,
        tensor.dtype,
    )

    high = torch.from_numpy(MS_HIGH).to(
        tensor.device,
        tensor.dtype,
    )

    return tensor * (high - low) + low


## Dataset وMisalignment Augmentation

In [ ]:

from torch.utils.data import Dataset, DataLoader


def shift_without_wrap(tensor, shift_y, shift_x):
    # tensor shape: C, H, W
    channels, height, width = tensor.shape

    pad_left = max(shift_x, 0)
    pad_right = max(-shift_x, 0)
    pad_top = max(shift_y, 0)
    pad_bottom = max(-shift_y, 0)

    padded = F.pad(
        tensor.unsqueeze(0),
        (
            pad_left,
            pad_right,
            pad_top,
            pad_bottom,
        ),
        mode="replicate",
    )[0]

    start_x = max(-shift_x, 0)
    start_y = max(-shift_y, 0)

    return padded[
        :,
        start_y:start_y + height,
        start_x:start_x + width,
    ]


class LocalDataset(Dataset):
    def __init__(self, folder, augment=False):
        self.files = sorted(Path(folder).glob("*.npz"))
        self.augment = augment

        if not self.files:
            raise RuntimeError(f"No files in {folder}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]

        with np.load(path) as file:
            lr_ms = torch.from_numpy(
                normalize_ms(file["lr_ms"].copy())
            ).float()

            pan = torch.from_numpy(
                normalize_pan(file["pan"].copy())
            ).float()

            target = torch.from_numpy(
                normalize_ms(file["target_ms"].copy())
            ).float()

        lms = F.interpolate(
            lr_ms.unsqueeze(0),
            size=pan.shape[-2:],
            mode="bicubic",
            align_corners=False,
        )[0].clamp(0, 1)

        if self.augment:
            if random.random() < 0.5:
                lms = torch.flip(lms, dims=[2])
                pan = torch.flip(pan, dims=[2])
                target = torch.flip(target, dims=[2])

            if random.random() < 0.5:
                lms = torch.flip(lms, dims=[1])
                pan = torch.flip(pan, dims=[1])
                target = torch.flip(target, dims=[1])

            rotations = random.randint(0, 3)

            if rotations:
                lms = torch.rot90(lms, rotations, dims=[1, 2])
                pan = torch.rot90(pan, rotations, dims=[1, 2])
                target = torch.rot90(target, rotations, dims=[1, 2])

            if (
                USE_MISALIGNMENT_AUGMENTATION
                and random.random() < MISALIGNMENT_PROBABILITY
            ):
                shift_x = random.randint(-MAX_SHIFT_HR, MAX_SHIFT_HR)
                shift_y = random.randint(-MAX_SHIFT_HR, MAX_SHIFT_HR)

                if shift_x != 0 or shift_y != 0:
                    pan = shift_without_wrap(
                        pan,
                        shift_y,
                        shift_x,
                    )

        return {
            "lr_ms": lr_ms,
            "lms": lms,
            "pan": pan,
            "target": target,
            "file": path.name,
        }


train_dataset = LocalDataset(TRAIN_DIR, augment=True)
val_dataset = LocalDataset(VAL_DIR, augment=False)
test_dataset = LocalDataset(TEST_DIR, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

assert len(train_dataset) == 231
assert len(val_dataset) == 22
assert len(test_dataset) == 22


## نقل الأوزان من 8 Bands إلى 6 Bands

In [ ]:

wv3_checkpoint = torch.load(
    WV3_ARNET_PATH,
    map_location="cpu",
    weights_only=False,
)

wv3_state = wv3_checkpoint["model_state_dict"]
wv3_epoch = int(wv3_checkpoint.get("epoch", 0))

source_model = ARNet(
    pan_channels=1,
    lms_channels=8,
)

source_model.load_state_dict(
    wv3_state,
    strict=True,
)

model = ARNet(
    pan_channels=1,
    lms_channels=6,
)

source_state = source_model.state_dict()
target_state = model.state_dict()

exact_transfers = 0

for key, value in source_state.items():
    if key in {
        "head_conv.weight",
        "tail_conv.weight",
        "tail_conv.bias",
    }:
        continue

    if (
        key in target_state
        and target_state[key].shape == value.shape
    ):
        target_state[key] = value.clone()
        exact_transfers += 1

# Head input order: PAN then MS bands.
source_head = source_state["head_conv.weight"]
target_head = target_state["head_conv.weight"]

target_head.zero_()
target_head[:, 0:1] = source_head[:, 0:1]

for local_band, source_band in enumerate(
    WV3_TO_LOCAL_BAND_MAPPING
):
    target_head[
        :,
        local_band + 1:local_band + 2,
    ] = source_head[
        :,
        source_band + 1:source_band + 2,
    ]

target_state["head_conv.weight"] = target_head
target_state["head_conv.bias"] = source_state[
    "head_conv.bias"
].clone()

target_state["tail_conv.weight"] = source_state[
    "tail_conv.weight"
][WV3_TO_LOCAL_BAND_MAPPING].clone()

target_state["tail_conv.bias"] = source_state[
    "tail_conv.bias"
][WV3_TO_LOCAL_BAND_MAPPING].clone()

model.load_state_dict(target_state, strict=True)
model = model.to(device)

print("WV3 epoch:", wv3_epoch)
print("Exact tensors:", exact_transfers)
print("Head:", source_head.shape, "→", model.head_conv.weight.shape)
print(
    "Tail:",
    source_state["tail_conv.weight"].shape,
    "→",
    model.tail_conv.weight.shape,
)


In [ ]:

reserved_kernels = {}

for name, module in model.named_modules():
    if hasattr(module, "reserved_NXY"):
        reserved_kernels[name] = (
            module.reserved_NXY
            .detach()
            .cpu()
            .tolist()
        )

print(json.dumps(reserved_kernels, indent=2))

if wv3_epoch < 100:
    print(
        "Warning: checkpoint epoch is below 100; "
        "some reserved kernels may remain 3x3."
    )


## Loss والمقاييس

In [ ]:

from skimage.metrics import structural_similarity


def charbonnier(prediction, target, epsilon=1e-3):
    difference = prediction - target

    return torch.sqrt(
        difference * difference + epsilon * epsilon
    ).mean()


def sam_cosine_loss(prediction, target):
    prediction_dn = tensor_to_dn(prediction)
    target_dn = tensor_to_dn(target)

    dot = torch.sum(
        prediction_dn * target_dn,
        dim=1,
    )

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction_dn, dim=1)
        * torch.linalg.vector_norm(target_dn, dim=1),
        min=1e-8,
    )

    cosine = torch.clamp(
        dot / denominator,
        -1,
        1,
    )

    return (1.0 - cosine).mean()


def total_loss(prediction, target):
    return (
        charbonnier(prediction, target)
        + 0.05 * sam_cosine_loss(prediction, target)
    )


def metric_values(prediction, target):
    mse = torch.mean(
        (prediction - target) ** 2,
        dim=(1, 2, 3),
    )

    psnr = 10.0 * torch.log10(
        1.0 / torch.clamp(mse, min=1e-12)
    )

    prediction_dn = tensor_to_dn(prediction)
    target_dn = tensor_to_dn(target)

    dot = torch.sum(
        prediction_dn * target_dn,
        dim=1,
    )

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction_dn, dim=1)
        * torch.linalg.vector_norm(target_dn, dim=1),
        min=1e-8,
    )

    sam = (
        torch.acos(
            torch.clamp(
                dot / denominator,
                -1 + 1e-7,
                1 - 1e-7,
            )
        )
        * 180.0
        / math.pi
    ).mean(dim=(1, 2))

    rmse = torch.sqrt(
        torch.mean(
            (prediction_dn - target_dn) ** 2,
            dim=(2, 3),
        )
    )

    target_mean = torch.mean(
        target_dn,
        dim=(2, 3),
    ).abs().clamp_min(1e-6)

    ergas = (
        100.0
        / 4.0
        * torch.sqrt(
            torch.mean(
                (rmse / target_mean) ** 2,
                dim=1,
            )
        )
    )

    return psnr, sam, ergas


def ssim_values(prediction, target):
    prediction_np = prediction.detach().float().cpu().numpy()
    target_np = target.detach().float().cpu().numpy()

    values = []

    for sample_index in range(prediction_np.shape[0]):
        band_scores = []

        for band_index in range(prediction_np.shape[1]):
            band_scores.append(
                structural_similarity(
                    target_np[sample_index, band_index],
                    prediction_np[sample_index, band_index],
                    data_range=1.0,
                )
            )

        values.append(float(np.mean(band_scores)))

    return values


In [ ]:

from tqdm.auto import tqdm


@torch.inference_mode()
def evaluate_model(loader):
    model.eval()

    results = {
        "loss": [],
        "psnr": [],
        "ssim": [],
        "sam": [],
        "ergas": [],
        "l1": [],
    }

    inference_epoch = max(101, wv3_epoch + 1)

    for batch in tqdm(
        loader,
        desc="ARNet evaluation",
        leave=False,
    ):
        lms = batch["lms"].to(device)
        pan = batch["pan"].to(device)
        target = batch["target"].to(device)

        prediction = model(
            pan,
            lms,
            epoch=inference_epoch,
            hw_range=HW_RANGE,
        ).clamp(0, 1)

        loss = total_loss(prediction, target)
        psnr, sam, ergas = metric_values(prediction, target)

        results["loss"].append(loss.item())
        results["psnr"].extend(psnr.cpu().tolist())
        results["sam"].extend(sam.cpu().tolist())
        results["ergas"].extend(ergas.cpu().tolist())
        results["ssim"].extend(ssim_values(prediction, target))
        results["l1"].append(F.l1_loss(prediction, target).item())

    return {
        key: float(np.mean(values))
        for key, values in results.items()
    }


## Stage A وStage B

In [ ]:

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau


def set_trainable(module, value):
    for parameter in module.parameters():
        parameter.requires_grad = value


def configure_stage_a():
    set_trainable(model, False)
    set_trainable(model.head_conv, True)
    set_trainable(model.tail_conv, True)


def configure_stage_b():
    set_trainable(model, True)


def current_trainable_parameters():
    return [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]


def run_stage(
    stage_name,
    epochs,
    learning_rate,
    history,
    epoch_offset,
):
    optimizer = AdamW(
        current_trainable_parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    best_validation_loss = float("inf")
    patience_counter = 0

    print("\n" + "=" * 72)
    print(stage_name)
    print(
        "Trainable:",
        f"{sum(p.numel() for p in current_trainable_parameters()):,}",
    )
    print("=" * 72)

    for local_epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        losses = []

        arconv_epoch = max(
            101,
            wv3_epoch + epoch_offset + local_epoch,
        )

        progress = tqdm(
            train_loader,
            desc=f"{stage_name} {local_epoch}/{epochs}",
            leave=False,
        )

        for batch_index, batch in enumerate(progress, start=1):
            lms = batch["lms"].to(device, non_blocking=True)
            pan = batch["pan"].to(device, non_blocking=True)
            target = batch["target"].to(device, non_blocking=True)

            prediction = model(
                pan,
                lms,
                epoch=arconv_epoch,
                hw_range=HW_RANGE,
            )

            loss = total_loss(prediction, target)

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss.")

            (loss / ACCUMULATION_STEPS).backward()

            should_step = (
                batch_index % ACCUMULATION_STEPS == 0
                or batch_index == len(train_loader)
            )

            if should_step:
                torch.nn.utils.clip_grad_norm_(
                    current_trainable_parameters(),
                    max_norm=1.0,
                )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            losses.append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.5f}")

        train_loss = float(np.mean(losses))
        validation = evaluate_model(val_loader)

        scheduler.step(validation["loss"])

        history.append(
            {
                "stage": stage_name,
                "local_epoch": local_epoch,
                "arconv_epoch": arconv_epoch,
                "train_loss": train_loss,
                "validation": validation,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )

        print(
            f"{stage_name} | "
            f"Epoch {local_epoch:02d}/{epochs} | "
            f"Train {train_loss:.5f} | "
            f"PSNR {validation['psnr']:.3f} | "
            f"SSIM {validation['ssim']:.4f} | "
            f"SAM {validation['sam']:.3f}° | "
            f"ERGAS {validation['ergas']:.3f}"
        )

        checkpoint = {
            "model_state_dict": model.state_dict(),
            "stage": stage_name,
            "local_epoch": local_epoch,
            "arconv_epoch": arconv_epoch,
            "wv3_checkpoint_epoch": wv3_epoch,
            "band_mapping": WV3_TO_LOCAL_BAND_MAPPING,
            "misalignment_augmentation": USE_MISALIGNMENT_AUGMENTATION,
            "validation": validation,
            "history": history,
        }

        torch.save(checkpoint, LAST_PATH)

        if validation["loss"] < best_validation_loss:
            best_validation_loss = validation["loss"]
            patience_counter = 0
            torch.save(checkpoint, BEST_PATH)
            print("Saved new best.")
        else:
            patience_counter += 1

        with open(HISTORY_PATH, "w", encoding="utf-8") as file:
            json.dump(
                history,
                file,
                indent=2,
                ensure_ascii=False,
            )

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

    return history


In [ ]:

history = []

configure_stage_a()

history = run_stage(
    stage_name="Stage A - Head and Tail",
    epochs=STAGE_A_EPOCHS,
    learning_rate=STAGE_A_LR,
    history=history,
    epoch_offset=0,
)

stage_a_checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    stage_a_checkpoint["model_state_dict"],
    strict=True,
)

configure_stage_b()

history = run_stage(
    stage_name="Stage B - Full Fine-tuning",
    epochs=STAGE_B_EPOCHS,
    learning_rate=STAGE_B_LR,
    history=history,
    epoch_offset=STAGE_A_EPOCHS,
)

print("Best:", BEST_PATH)


## Test النهائي والمقارنة

In [ ]:

best_checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"],
    strict=True,
)

model.eval()

method_names = [
    "Bicubic",
    "Fusion CNN",
    "WV3-Transfer HAT",
    "WV3-Transfer ARNet",
]

values = {
    name: {
        "psnr": [],
        "ssim": [],
        "sam": [],
        "ergas": [],
        "l1": [],
    }
    for name in method_names
}

inference_epoch = max(
    101,
    int(best_checkpoint.get("arconv_epoch", 101)),
)

with torch.inference_mode():
    for batch in tqdm(
        test_loader,
        desc="Final local Test",
    ):
        lms = batch["lms"].to(device)
        pan = batch["pan"].to(device)
        target = batch["target"].to(device)

        arnet = model(
            pan,
            lms,
            epoch=inference_epoch,
            hw_range=HW_RANGE,
        ).clamp(0, 1)

        file_name = batch["file"][0]
        expert_path = EXPERT_TEST_CACHE / file_name

        with np.load(expert_path) as experts:
            hat = torch.from_numpy(
                experts["hat"].astype(np.float32)
            ).unsqueeze(0).to(device)

            cnn = torch.from_numpy(
                experts["cnn"].astype(np.float32)
            ).unsqueeze(0).to(device)

        predictions = {
            "Bicubic": lms,
            "Fusion CNN": cnn,
            "WV3-Transfer HAT": hat,
            "WV3-Transfer ARNet": arnet,
        }

        for name, prediction in predictions.items():
            psnr, sam, ergas = metric_values(
                prediction,
                target,
            )

            values[name]["psnr"].extend(psnr.cpu().tolist())
            values[name]["sam"].extend(sam.cpu().tolist())
            values[name]["ergas"].extend(ergas.cpu().tolist())
            values[name]["ssim"].extend(
                ssim_values(prediction, target)
            )
            values[name]["l1"].append(
                F.l1_loss(prediction, target).item()
            )

rows = []

for name in method_names:
    rows.append(
        {
            "Method": name,
            "PSNR": float(np.mean(values[name]["psnr"])),
            "SSIM": float(np.mean(values[name]["ssim"])),
            "SAM": float(np.mean(values[name]["sam"])),
            "ERGAS": float(np.mean(values[name]["ergas"])),
            "L1": float(np.mean(values[name]["l1"])),
        }
    )

table = pd.DataFrame(rows)

table.to_csv(
    TEST_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

payload = {
    "protocol": (
        "Spatial local Train/Validation/Test; "
        "ARNet transferred from PanCollection WV3"
    ),
    "wv3_checkpoint_epoch": wv3_epoch,
    "band_mapping": WV3_TO_LOCAL_BAND_MAPPING,
    "misalignment_augmentation": USE_MISALIGNMENT_AUGMENTATION,
    "metrics": rows,
}

with open(TEST_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(
        payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

display(table.round(6))

print("Saved:", TEST_CSV_PATH)
print("Saved:", TEST_JSON_PATH)



# القرار بعد النتائج

لا تبدأ دمج HAT + ARNet قبل إرسال جدول Test.

ننتقل إلى Spatial OOF Fusion فقط عندما يكون ARNet:

```text
أفضل من CNN في مقياس واحد على الأقل
أو
قريبًا من HAT مع نمط أخطاء مختلف
```

أرسل:

```text
WV3 checkpoint epoch
Reserved kernels
Stage A results
Stage B results
Final Test table
```
